# Scenario Analysis

This notebook provides tools to review and analyze injected supply chain scenarios.

**Supported Scenarios:**
| Scenario | Mvmt Type | Description |
|----------|-----------|-------------|
| SCN001 | 344 | Stock deviation/adjustment |
| SCN002 | 551 | Contamination (scrapping) |
| SCN003 | 551 | Fire damage (scrapping) |
| SCN008 | 311 | Transfer posting |
| SCN009 | 101 | Goods receipt reroute |

In [ ]:
# Configuration
dbutils.widgets.text("CATALOG", "sample_synthetic_sap", "Catalog Name")
dbutils.widgets.text("SCHEMA", "sap", "Schema Name")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")

print(f"Analyzing scenarios in {CATALOG}.{SCHEMA}")

## 1. Scenario Metadata

View all injected scenarios and their configurations.

In [ ]:
# Load scenario metadata
try:
    scenarios_df = spark.table(f"{CATALOG}.{SCHEMA}.scenario_metadata")
    print(f"Found {scenarios_df.count()} injected scenarios:")
    display(scenarios_df)
except Exception as e:
    print(f"No scenario metadata found. Scenarios may not have been injected yet.")
    print(f"Run the 'Inject Scenarios' notebook first with scenarios enabled.")

## 2. Scenario MATDOC Records

View the actual MATDOC (material document) records created by scenario injection.

In [ ]:
# View scenario MATDOC records
matdoc_scenarios = spark.sql(f"""
    SELECT 
        MBLNR as Document_Number,
        BWART as Movement_Type,
        MATNR as Material,
        WERKS as Plant,
        LGORT as Storage_Location,
        UMLGO as Dest_Location,
        SHKZG as Debit_Credit,
        MENGE as Quantity,
        BUDAT as Posting_Date,
        BKTXT as Description
    FROM {CATALOG}.{SCHEMA}.matdoc
    WHERE MBLNR LIKE 'SCN%'
    ORDER BY MBLNR
""")

print(f"Found {matdoc_scenarios.count()} scenario MATDOC records:")
display(matdoc_scenarios)

## 3. Inventory Impact Summary

Aggregate view of inventory impact by scenario and movement type.

In [ ]:
# Inventory impact by scenario
impact_summary = spark.sql(f"""
    SELECT
        SUBSTRING(MBLNR, 1, 6) as Scenario,
        BWART as Movement_Type,
        COUNT(*) as Record_Count,
        SUM(CASE WHEN SHKZG = 'H' THEN -CAST(MENGE AS DOUBLE) ELSE CAST(MENGE AS DOUBLE) END) as Net_Qty_Impact,
        COUNT(DISTINCT MATNR) as Materials_Affected,
        COUNT(DISTINCT WERKS) as Plants_Affected
    FROM {CATALOG}.{SCHEMA}.matdoc
    WHERE MBLNR LIKE 'SCN%'
    GROUP BY SUBSTRING(MBLNR, 1, 6), BWART
    ORDER BY Scenario, Movement_Type
""")

print("Inventory Impact Summary:")
display(impact_summary)

## 4. Affected Materials and Locations

Detailed view of which materials and locations were affected by each scenario.

In [ ]:
# Affected materials and plants
affected = spark.sql(f"""
    SELECT DISTINCT
        SUBSTRING(MBLNR, 1, 6) as Scenario,
        BKTXT as Description,
        MATNR as Material,
        WERKS as Plant,
        LGORT as Storage_Location,
        BWART as Movement_Type
    FROM {CATALOG}.{SCHEMA}.matdoc
    WHERE MBLNR LIKE 'SCN%'
    ORDER BY Scenario, Material, Plant
""")

print("Affected Materials and Locations:")
display(affected)

## 5. Current Inventory Levels (MARD)

View current inventory levels for materials affected by scenarios.

In [ ]:
# Current inventory for affected materials
affected_inventory = spark.sql(f"""
    SELECT 
        d.MATNR as Material,
        d.WERKS as Plant,
        d.LGORT as Storage_Location,
        d.LABST as Current_Stock,
        m.MBLNR as Scenario_Doc,
        m.BWART as Movement_Type,
        CASE WHEN m.SHKZG = 'H' THEN -CAST(m.MENGE AS DOUBLE) ELSE CAST(m.MENGE AS DOUBLE) END as Scenario_Impact
    FROM {CATALOG}.{SCHEMA}.mard d
    INNER JOIN (
        SELECT DISTINCT MATNR, WERKS, LGORT, MBLNR, BWART, SHKZG, MENGE
        FROM {CATALOG}.{SCHEMA}.matdoc
        WHERE MBLNR LIKE 'SCN%'
    ) m ON d.MATNR = m.MATNR AND d.WERKS = m.WERKS AND d.LGORT = m.LGORT
    ORDER BY d.MATNR, d.WERKS, d.LGORT
""")

print("Current Inventory for Scenario-Affected Materials:")
display(affected_inventory)

## 6. Historical Stock Impact (MARDH)

View how scenarios impacted historical stock records.

In [ ]:
# Historical stock for affected materials
historical_stock = spark.sql(f"""
    SELECT 
        h.MATNR as Material,
        h.WERKS as Plant,
        h.LGORT as Storage_Location,
        h.LFGJA as Year,
        h.LFMON as Month,
        h.LABST as Period_End_Stock
    FROM {CATALOG}.{SCHEMA}.mardh h
    WHERE EXISTS (
        SELECT 1 FROM {CATALOG}.{SCHEMA}.matdoc m
        WHERE m.MBLNR LIKE 'SCN%'
        AND m.MATNR = h.MATNR AND m.WERKS = h.WERKS AND m.LGORT = h.LGORT
    )
    ORDER BY h.MATNR, h.WERKS, h.LGORT, h.LFGJA, h.LFMON
""")

print("Historical Stock for Scenario-Affected Materials:")
display(historical_stock)

## 7. Movement Type Analysis

Breakdown of all movement types in MATDOC, highlighting scenario-related movements.

In [ ]:
# Movement type distribution
movement_analysis = spark.sql(f"""
    SELECT 
        BWART as Movement_Type,
        CASE 
            WHEN BWART = '101' THEN 'Goods Receipt'
            WHEN BWART = '102' THEN 'GR Reversal'
            WHEN BWART = '261' THEN 'Goods Issue (Production)'
            WHEN BWART = '311' THEN 'Transfer Posting'
            WHEN BWART = '344' THEN 'Stock Adjustment'
            WHEN BWART = '551' THEN 'Scrapping'
            WHEN BWART = '601' THEN 'Goods Issue (Delivery)'
            WHEN BWART = '641' THEN 'GI Reversal'
            ELSE 'Other'
        END as Description,
        COUNT(*) as Total_Records,
        SUM(CASE WHEN MBLNR LIKE 'SCN%' THEN 1 ELSE 0 END) as Scenario_Records,
        SUM(CASE WHEN MBLNR NOT LIKE 'SCN%' THEN 1 ELSE 0 END) as Normal_Records,
        SUM(CAST(MENGE AS DOUBLE)) as Total_Quantity
    FROM {CATALOG}.{SCHEMA}.matdoc
    GROUP BY BWART
    ORDER BY Total_Records DESC
""")

print("Movement Type Analysis:")
display(movement_analysis)

## 8. Scenario Verification Queries

Quick verification queries to validate scenario injection.

In [ ]:
# Count of scenario records by type
print("Scenario Records Count:")
scenario_count = spark.sql(f"""
    SELECT 
        'Total MATDOC Records' as Metric,
        COUNT(*) as Count
    FROM {CATALOG}.{SCHEMA}.matdoc
    UNION ALL
    SELECT 
        'Scenario MATDOC Records' as Metric,
        COUNT(*) as Count
    FROM {CATALOG}.{SCHEMA}.matdoc
    WHERE MBLNR LIKE 'SCN%'
    UNION ALL
    SELECT 
        'Scenarios in Metadata' as Metric,
        COUNT(*) as Count
    FROM {CATALOG}.{SCHEMA}.scenario_metadata
""")
display(scenario_count)

In [ ]:
# Validate scenario integrity
print("\nScenario Integrity Check:")
integrity_check = spark.sql(f"""
    SELECT
        'Materials in Scenario MATDOC exist in MARA' as Check,
        CASE 
            WHEN COUNT(*) = 0 THEN 'PASS'
            ELSE 'FAIL: ' || COUNT(*) || ' orphan records'
        END as Result
    FROM {CATALOG}.{SCHEMA}.matdoc m
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON m.MATNR = a.MATNR
    WHERE m.MBLNR LIKE 'SCN%' AND a.MATNR IS NULL
    
    UNION ALL
    
    SELECT
        'Scenario impacts reflected in MARD' as Check,
        CASE 
            WHEN COUNT(*) > 0 THEN 'PASS: ' || COUNT(*) || ' MARD records updated'
            ELSE 'WARNING: No matching MARD records'
        END as Result
    FROM {CATALOG}.{SCHEMA}.mard d
    WHERE EXISTS (
        SELECT 1 FROM {CATALOG}.{SCHEMA}.matdoc m
        WHERE m.MBLNR LIKE 'SCN%'
        AND m.MATNR = d.MATNR AND m.WERKS = d.WERKS AND m.LGORT = d.LGORT
    )
""")
display(integrity_check)

---

## 9. Fire Scenario (SCN003) Deep Dive

Detailed analysis of the fire damage scenario including stock losses, financial impact, and affected customer orders.

In [ ]:
# Fire Scenario Overview
fire_scenario = spark.sql(f"""
    SELECT 
        scenario_id,
        description,
        plant,
        storage_loc,
        quantity as total_qty_scrapped,
        downtime_days,
        recovery_date,
        injected_at
    FROM {CATALOG}.{SCHEMA}.scenario_metadata
    WHERE scenario_id = 'SCN003'
""")

if fire_scenario.count() > 0:
    fire_info = fire_scenario.collect()[0]
    print("=" * 70)
    print("FIRE SCENARIO (SCN003) SUMMARY")
    print("=" * 70)
    print(f"Description:      {fire_info['description']}")
    print(f"Plant:            {fire_info['plant']}")
    print(f"Storage Location: {fire_info['storage_loc']}")
    print(f"Downtime:         {fire_info['downtime_days']} days")
    print(f"Recovery Date:    {fire_info['recovery_date']}")
    print(f"Total Qty Lost:   {fire_info['total_qty_scrapped']:,.0f} units")
    print("=" * 70)
    display(fire_scenario)
else:
    print("No fire scenario (SCN003) found. Enable it in the pipeline configuration.")

### 9.1 Stock Lost by Product (Volume)

Breakdown of inventory destroyed in the fire by material.

In [ ]:
# Stock lost by product (volume) - Fire scenario ONLY
import matplotlib.pyplot as plt
import pandas as pd

fire_stock_loss = spark.sql(f"""
    SELECT 
        m.MATNR as Material,
        t.MAKTX as Material_Description,
        m.LGORT as Storage_Location,
        CAST(m.MENGE AS DOUBLE) as Quantity_Lost,
        a.MTART as Material_Type
    FROM {CATALOG}.{SCHEMA}.matdoc m
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON m.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON m.MATNR = a.MATNR
    WHERE m.MBLNR LIKE 'SCN003%'
    AND m.BWART = '551'
    ORDER BY CAST(m.MENGE AS DOUBLE) DESC
""")

df_loss = fire_stock_loss.toPandas()

if len(df_loss) > 0:
    print(f"Materials destroyed in fire: {len(df_loss)}")
    print(f"Total quantity lost: {df_loss['Quantity_Lost'].sum():,.0f} units")
    display(fire_stock_loss)
    
    # Dynamic figure height based on number of materials
    fig_height = max(6, len(df_loss) * 0.5)
    fig, ax = plt.subplots(figsize=(12, fig_height))
    
    # Sort by quantity for display
    df_loss = df_loss.sort_values('Quantity_Lost', ascending=True)
    
    # Shorter labels to avoid truncation
    df_loss['Label'] = df_loss['Material']
    
    colors = ['#d62728' if mt == 'FERT' else '#1f77b4' for mt in df_loss['Material_Type']]
    bars = ax.barh(df_loss['Label'], df_loss['Quantity_Lost'], color=colors, height=0.7)
    
    ax.set_xlabel('Quantity Lost (Units)', fontsize=12)
    ax.set_title('Fire Damage (SCN003): Stock Lost by Product', fontsize=14, fontweight='bold')
    
    # Add value labels on bars
    for bar, val in zip(bars, df_loss['Quantity_Lost']):
        ax.text(bar.get_width() + max(df_loss['Quantity_Lost']) * 0.01, 
                bar.get_y() + bar.get_height()/2, 
                f'{val:,.0f}', va='center', fontsize=9)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#d62728', label='Finished Goods (FERT)'),
                       Patch(facecolor='#1f77b4', label='Raw Materials (ROH)')]
    ax.legend(handles=legend_elements, loc='lower right')
    
    # Adjust margins
    ax.set_xlim(0, max(df_loss['Quantity_Lost']) * 1.15)
    plt.tight_layout()
    plt.show()
else:
    print("No fire damage records found (SCN003).")

### 9.2 Stock Lost by Product (Value)

Financial impact of destroyed inventory using standard cost from MBEW.

In [ ]:
# Stock lost by product (value) - Fire scenario with financial impact
fire_value_loss = spark.sql(f"""
    SELECT 
        m.MATNR as Material,
        t.MAKTX as Material_Description,
        a.MTART as Material_Type,
        m.LGORT as Storage_Location,
        CAST(m.MENGE AS DOUBLE) as Quantity_Lost,
        COALESCE(b.STPRS, 0) as Standard_Price,
        COALESCE(b.PEINH, 1) as Price_Unit,
        ROUND(CAST(m.MENGE AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1), 2) as Value_Lost
    FROM {CATALOG}.{SCHEMA}.matdoc m
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON m.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON m.MATNR = a.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON m.MATNR = b.MATNR AND m.WERKS = b.BWKEY
    WHERE m.MBLNR LIKE 'SCN003%'
    AND m.BWART = '551'
    ORDER BY CAST(m.MENGE AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1) DESC
""")

df_value = fire_value_loss.toPandas()

if len(df_value) > 0:
    total_value_lost = df_value['Value_Lost'].sum()
    total_qty_lost = df_value['Quantity_Lost'].sum()
    
    print("=" * 70)
    print(f"TOTAL STOCK LOSS FROM FIRE")
    print("=" * 70)
    print(f"Materials destroyed:  {len(df_value)}")
    print(f"Total units lost:     {total_qty_lost:,.0f}")
    print(f"Total value lost:     £{total_value_lost:,.2f}")
    print("=" * 70)
    display(fire_value_loss)
    
    # Dynamic figure height
    fig_height = max(6, len(df_value) * 0.5)
    fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))
    
    # Sort by value
    df_value_sorted = df_value.sort_values('Value_Lost', ascending=True)
    df_value_sorted['Label'] = df_value_sorted['Material']
    
    # Chart 1: Volume (Quantity Lost)
    colors_vol = ['#d62728' if mt == 'FERT' else '#1f77b4' for mt in df_value_sorted['Material_Type']]
    axes[0].barh(df_value_sorted['Label'], df_value_sorted['Quantity_Lost'], color=colors_vol, height=0.7)
    axes[0].set_xlabel('Quantity Lost (Units)', fontsize=11)
    axes[0].set_title('Volume Lost', fontsize=12, fontweight='bold')
    axes[0].set_xlim(0, max(df_value_sorted['Quantity_Lost']) * 1.15)
    
    # Chart 2: Value (£)
    colors_val = ['#d62728' if mt == 'FERT' else '#1f77b4' for mt in df_value_sorted['Material_Type']]
    axes[1].barh(df_value_sorted['Label'], df_value_sorted['Value_Lost'], color=colors_val, height=0.7)
    axes[1].set_xlabel('Value Lost (£)', fontsize=11)
    axes[1].set_title('Financial Loss', fontsize=12, fontweight='bold')
    axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'£{x:,.0f}'))
    axes[1].set_xlim(0, max(df_value_sorted['Value_Lost']) * 1.15)
    
    fig.suptitle('Fire Damage (SCN003): Stock Loss Analysis', fontsize=14, fontweight='bold')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#d62728', label='Finished Goods (FERT)'),
                       Patch(facecolor='#1f77b4', label='Raw Materials (ROH)')]
    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, 0.98))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()
    
    # Summary by material type
    print("\nLoss Summary by Material Type:")
    summary = df_value.groupby('Material_Type').agg({
        'Material': 'count',
        'Quantity_Lost': 'sum',
        'Value_Lost': 'sum'
    }).rename(columns={'Material': 'Materials_Count'})
    print(summary.to_string())
else:
    print("No fire damage records found (SCN003).")

### 9.3 Customer Orders Impacted

Analysis of customer orders affected by the fire - orders that could not be fulfilled due to stock destruction or plant downtime.

In [ ]:
# Get fire scenario details
fire_details = spark.sql(f"""
    SELECT plant, recovery_date, downtime_days
    FROM {CATALOG}.{SCHEMA}.scenario_metadata
    WHERE scenario_id = 'SCN003'
""").collect()

if len(fire_details) > 0:
    fire_plant = fire_details[0]['plant']
    recovery_date = fire_details[0]['recovery_date']
    
    # Find ONLY orders that were NOT delivered (truly impacted by fire)
    # These are orders at the fire plant, for fire-destroyed materials, that have no delivery
    impacted_orders = spark.sql(f"""
        WITH fire_materials AS (
            SELECT DISTINCT MATNR, WERKS
            FROM {CATALOG}.{SCHEMA}.matdoc
            WHERE MBLNR LIKE 'SCN003%'
            AND BWART = '551'
        ),
        delivered_orders AS (
            SELECT DISTINCT f.VBELN
            FROM {CATALOG}.{SCHEMA}.vbfa f
            INNER JOIN {CATALOG}.{SCHEMA}.likp d ON f.VBELN_N = d.VBELN
        )
        SELECT 
            o.VBELN as Order_Number,
            o.KUNNR as Customer,
            c.NAME1 as Customer_Name,
            p.MATNR as Material,
            t.MAKTX as Material_Description,
            p.WERKS as Plant,
            CAST(p.KWMENG AS DOUBLE) as Order_Qty,
            CAST(p.NETWR AS DOUBLE) as Line_Value,
            e.EDATU as Requested_Delivery_Date
        FROM {CATALOG}.{SCHEMA}.vbak o
        INNER JOIN {CATALOG}.{SCHEMA}.vbap p ON o.VBELN = p.VBELN
        INNER JOIN fire_materials fm ON p.MATNR = fm.MATNR AND p.WERKS = fm.WERKS
        LEFT JOIN {CATALOG}.{SCHEMA}.vbep e ON p.VBELN = e.VBELN AND p.POSNR = e.POSNR
        LEFT JOIN {CATALOG}.{SCHEMA}.kna1 c ON o.KUNNR = c.KUNNR
        LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON p.MATNR = t.MATNR
        WHERE o.VBELN NOT IN (SELECT VBELN FROM delivered_orders)
        ORDER BY CAST(p.NETWR AS DOUBLE) DESC
    """)
    
    df_orders = impacted_orders.toPandas()
    
    if len(df_orders) > 0:
        total_orders = df_orders['Order_Number'].nunique()
        total_customers = df_orders['Customer'].nunique()
        total_value = df_orders['Line_Value'].sum()
        total_qty = df_orders['Order_Qty'].sum()
        
        print("=" * 70)
        print("CUSTOMER ORDERS NOT FULFILLED DUE TO FIRE")
        print("=" * 70)
        print(f"Plant affected:           {fire_plant}")
        print(f"Orders not delivered:     {total_orders}")
        print(f"Customers impacted:       {total_customers}")
        print(f"Total qty not delivered:  {total_qty:,.0f} units")
        print(f"Revenue lost:             £{total_value:,.2f}")
        print("=" * 70)
        
        display(impacted_orders)
    else:
        print(f"No unfulfilled orders found for fire-affected materials at plant {fire_plant}")
        print("(All orders for these materials may have been delivered before the fire)")
else:
    print("No fire scenario found.")

In [ ]:
# Customer impact visualization - only unfulfilled orders
if len(fire_details) > 0 and len(df_orders) > 0:
    # Dynamic sizing
    num_customers = df_orders['Customer'].nunique()
    fig_height = max(5, min(num_customers * 0.4, 12))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))
    
    # Chart 1: Revenue Lost by Customer
    customer_impact = df_orders.groupby(['Customer', 'Customer_Name']).agg({
        'Order_Number': 'nunique',
        'Line_Value': 'sum'
    }).reset_index()
    customer_impact = customer_impact.sort_values('Line_Value', ascending=True)
    customer_impact['Label'] = customer_impact['Customer']
    
    axes[0].barh(customer_impact['Label'], customer_impact['Line_Value'], color='#d62728', height=0.7)
    axes[0].set_xlabel('Revenue Lost (£)', fontsize=11)
    axes[0].set_title('Revenue Lost by Customer', fontsize=12, fontweight='bold')
    axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'£{x:,.0f}'))
    
    # Chart 2: Orders Lost by Material
    material_impact = df_orders.groupby('Material').agg({
        'Order_Number': 'nunique',
        'Line_Value': 'sum',
        'Order_Qty': 'sum'
    }).reset_index()
    material_impact = material_impact.sort_values('Line_Value', ascending=True)
    
    axes[1].barh(material_impact['Material'], material_impact['Line_Value'], color='#ff7f0e', height=0.7)
    axes[1].set_xlabel('Revenue Lost (£)', fontsize=11)
    axes[1].set_title('Revenue Lost by Product', fontsize=12, fontweight='bold')
    axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'£{x:,.0f}'))
    
    fig.suptitle('Fire Damage (SCN003): Unfulfilled Customer Orders', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()
    
    # Summary tables
    print("\nRevenue Lost by Customer:")
    customer_summary = df_orders.groupby(['Customer', 'Customer_Name']).agg({
        'Order_Number': 'nunique',
        'Order_Qty': 'sum',
        'Line_Value': 'sum'
    }).reset_index().sort_values('Line_Value', ascending=False)
    customer_summary.columns = ['Customer', 'Name', 'Orders', 'Qty', 'Value_Lost']
    display(spark.createDataFrame(customer_summary))
else:
    if len(fire_details) > 0:
        print("No unfulfilled orders to visualize.")

### 9.4 Fire Impact Executive Summary

Consolidated dashboard showing total business impact from the fire scenario.

In [ ]:
# Executive Summary Dashboard
if len(fire_details) > 0:
    fire_plant = fire_details[0]['plant']
    downtime_days = fire_details[0]['downtime_days']
    recovery_date = fire_details[0]['recovery_date']
    
    # Stock loss metrics (from fire MATDOC records only)
    stock_metrics = spark.sql(f"""
        SELECT 
            COUNT(DISTINCT m.MATNR) as materials_destroyed,
            SUM(CAST(m.MENGE AS DOUBLE)) as total_qty_lost,
            SUM(CAST(m.MENGE AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1)) as total_value_lost
        FROM {CATALOG}.{SCHEMA}.matdoc m
        LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON m.MATNR = b.MATNR AND m.WERKS = b.BWKEY
        WHERE m.MBLNR LIKE 'SCN003%'
        AND m.BWART = '551'
    """).collect()[0]
    
    # Customer order metrics - ONLY unfulfilled orders
    try:
        order_metrics = spark.sql(f"""
            WITH fire_materials AS (
                SELECT DISTINCT MATNR, WERKS FROM {CATALOG}.{SCHEMA}.matdoc
                WHERE MBLNR LIKE 'SCN003%' AND BWART = '551'
            ),
            delivered_orders AS (
                SELECT DISTINCT f.VBELN
                FROM {CATALOG}.{SCHEMA}.vbfa f
                INNER JOIN {CATALOG}.{SCHEMA}.likp d ON f.VBELN_N = d.VBELN
            )
            SELECT 
                COUNT(DISTINCT o.VBELN) as orders_not_fulfilled,
                COUNT(DISTINCT o.KUNNR) as customers_affected,
                SUM(CAST(p.NETWR AS DOUBLE)) as revenue_lost
            FROM {CATALOG}.{SCHEMA}.vbak o
            INNER JOIN {CATALOG}.{SCHEMA}.vbap p ON o.VBELN = p.VBELN
            INNER JOIN fire_materials fm ON p.MATNR = fm.MATNR AND p.WERKS = fm.WERKS
            WHERE o.VBELN NOT IN (SELECT VBELN FROM delivered_orders)
        """).collect()[0]
    except:
        order_metrics = {'orders_not_fulfilled': 0, 'customers_affected': 0, 'revenue_lost': 0}
    
    # Create executive dashboard
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'FIRE SCENARIO (SCN003) - EXECUTIVE DASHBOARD\nPlant {fire_plant} | {downtime_days} Days Downtime | Recovery: {recovery_date}', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    red = '#d62728'
    orange = '#ff7f0e'
    
    # ROW 1: Stock Loss
    # KPI 1: Materials Destroyed
    axes[0, 0].text(0.5, 0.5, f"{stock_metrics['materials_destroyed']}", 
                    ha='center', va='center', fontsize=48, fontweight='bold', color=red)
    axes[0, 0].text(0.5, 0.15, "Materials\nDestroyed", ha='center', va='center', fontsize=14)
    axes[0, 0].set_xlim(0, 1); axes[0, 0].set_ylim(0, 1)
    axes[0, 0].axis('off'); axes[0, 0].set_facecolor('#fff5f5')
    
    # KPI 2: Quantity Lost
    qty_lost = stock_metrics['total_qty_lost'] or 0
    axes[0, 1].text(0.5, 0.5, f"{qty_lost:,.0f}", 
                    ha='center', va='center', fontsize=42, fontweight='bold', color=red)
    axes[0, 1].text(0.5, 0.15, "Units\nDestroyed", ha='center', va='center', fontsize=14)
    axes[0, 1].set_xlim(0, 1); axes[0, 1].set_ylim(0, 1)
    axes[0, 1].axis('off'); axes[0, 1].set_facecolor('#fff5f5')
    
    # KPI 3: Stock Value Lost
    value_lost = stock_metrics['total_value_lost'] or 0
    axes[0, 2].text(0.5, 0.5, f"£{value_lost:,.0f}", 
                    ha='center', va='center', fontsize=36, fontweight='bold', color=red)
    axes[0, 2].text(0.5, 0.15, "Stock Value\nLost", ha='center', va='center', fontsize=14)
    axes[0, 2].set_xlim(0, 1); axes[0, 2].set_ylim(0, 1)
    axes[0, 2].axis('off'); axes[0, 2].set_facecolor('#fff5f5')
    
    # ROW 2: Customer Impact (unfulfilled orders only)
    # KPI 4: Orders Not Fulfilled
    orders = order_metrics['orders_not_fulfilled'] or 0
    axes[1, 0].text(0.5, 0.5, f"{orders:,}", 
                    ha='center', va='center', fontsize=48, fontweight='bold', color=orange)
    axes[1, 0].text(0.5, 0.15, "Orders Not\nFulfilled", ha='center', va='center', fontsize=14)
    axes[1, 0].set_xlim(0, 1); axes[1, 0].set_ylim(0, 1)
    axes[1, 0].axis('off'); axes[1, 0].set_facecolor('#fff8f0')
    
    # KPI 5: Customers Affected
    customers = order_metrics['customers_affected'] or 0
    axes[1, 1].text(0.5, 0.5, f"{customers:,}", 
                    ha='center', va='center', fontsize=48, fontweight='bold', color=orange)
    axes[1, 1].text(0.5, 0.15, "Customers\nAffected", ha='center', va='center', fontsize=14)
    axes[1, 1].set_xlim(0, 1); axes[1, 1].set_ylim(0, 1)
    axes[1, 1].axis('off'); axes[1, 1].set_facecolor('#fff8f0')
    
    # KPI 6: Revenue Lost
    revenue_lost = order_metrics['revenue_lost'] or 0
    axes[1, 2].text(0.5, 0.5, f"£{revenue_lost:,.0f}", 
                    ha='center', va='center', fontsize=36, fontweight='bold', color=orange)
    axes[1, 2].text(0.5, 0.15, "Revenue\nLost", ha='center', va='center', fontsize=14)
    axes[1, 2].set_xlim(0, 1); axes[1, 2].set_ylim(0, 1)
    axes[1, 2].axis('off'); axes[1, 2].set_facecolor('#fff8f0')
    
    plt.tight_layout()
    plt.show()
    
    # Text summary
    print("\n" + "=" * 70)
    print("FIRE SCENARIO IMPACT SUMMARY")
    print("=" * 70)
    print(f"{'Metric':<35} {'Value':>20}")
    print("-" * 55)
    print(f"{'Plant Affected:':<35} {fire_plant:>20}")
    print(f"{'Downtime Duration:':<35} {str(downtime_days) + ' days':>20}")
    print(f"{'Recovery Date:':<35} {recovery_date:>20}")
    print("-" * 55)
    print(f"{'STOCK LOSS':<35}")
    print(f"{'  Materials Destroyed:':<35} {stock_metrics['materials_destroyed']:>20,}")
    print(f"{'  Total Units Lost:':<35} {qty_lost:>20,.0f}")
    print(f"{'  Stock Value Lost:':<35} {'£' + f'{value_lost:,.2f}':>19}")
    print("-" * 55)
    print(f"{'CUSTOMER IMPACT (Unfulfilled)':<35}")
    print(f"{'  Orders Not Fulfilled:':<35} {orders:>20,}")
    print(f"{'  Customers Affected:':<35} {customers:>20,}")
    print(f"{'  Revenue Lost:':<35} {'£' + f'{revenue_lost:,.2f}':>19}")
    print("=" * 70)
else:
    print("No fire scenario (SCN003) found.")